In [0]:
# %load_ext autoreload
# %autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
# schema_mappings = [ 
#   {
#   "field": "UVHeader",
#   "file_path": "/Volumes/users/matthew_moorcroft/mifir/lseg_xsd/Files_unavista_header_001_001_001.xsd",
#   },
#   {
#   "field": "BizAppHeader",
#   "file_path": "/Volumes/users/matthew_moorcroft/mifir/lseg_xsd/Files_BizAppheader_001_001_01.xsd"
#   },
#   {
#   "field": "Document",
#   "file_path": "/Volumes/users/matthew_moorcroft/mifir/lseg_xsd/PayloadDocument_auth_016_001_01_ESMAUG_Reporting_1_1_0.xsd",
#   "payload": True
#   }
# ]

In [0]:
# schema_mappings = [ 
#   {
#   "field": "Hdr",
#   "file_path": "/Volumes/users/matthew_moorcroft/cbi/xsd/test/2 BusinessApplicationHeader 001.json",
#   },
#   {
#   "field": "Pyld",
#   "file_path": "/Volumes/users/matthew_moorcroft/cbi/xsd/test/EMIR_Refit-_Outgoing_auth_107_001_01_ESMAUG_DATTSR_1_1_0.json",
#   "payload": True
#   }
# ]

In [ ]:
# Schema Creation Job Parameters
# This notebook creates JSON schemas and XSD files for XML processing

# Declare parameters
dbutils.widgets.text("schemas_path", "/Volumes/esma/default/regulatory_data/emir/schemas/")
dbutils.widgets.text("master_xsd_path", "/Volumes/esma/default/regulatory_data/emir/xsd/master_schema.xsd")
dbutils.widgets.text("payload_xsd_path", "/Volumes/esma/default/regulatory_data/emir/xsd/payload_schema.xsd")
dbutils.widgets.text("row_tag", "Stat")
dbutils.widgets.text("schema_mappings_json", '[{"field": "Hdr", "file_path": "/path/to/header.xsd"}, {"field": "Pyld", "file_path": "/path/to/payload.xsd", "payload": true}]')


In [0]:
import json

schemas_path = dbutils.widgets.get("schemas_path")
master_xsd_path = dbutils.widgets.get("master_xsd_path") 
payload_xsd_path = dbutils.widgets.get("payload_xsd_path")
row_tag = dbutils.widgets.get("row_tag")
schema_mappings_json = dbutils.widgets.get("schema_mappings_json")

# Parse schema mappings from JSON string
schema_mappings = json.loads(schema_mappings_json)

print(f"Schemas output path: {schemas_path}")
print(f"Master XSD path: {master_xsd_path}")
print(f"Payload XSD path: {payload_xsd_path}")  
print(f"Row tag: {row_tag}")
print(f"Using schema mappings from job parameters:")
for mapping in schema_mappings:
    print(f"  - {mapping['field']}: {mapping['file_path']}")
    if mapping.get('payload'):
        print(f"    (Payload field)")
        payloadXsdPath = mapping['file_path']

In [0]:
%scala
// Get widget values
val masterXsdPath = dbutils.widgets.get("master_xsd_path")
val schemaPath = dbutils.widgets.get("schemas_path") 
val payloadXsdPath = dbutils.widgets.get("payload_xsd_path")


In [0]:
%scala
import org.apache.spark.sql.types._
import org.apache.spark.sql.execution.datasources.xml.XSDToSchema
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

// Function to convert XSD to JSON schema
def convertXsdToJson(xsdPath: String, outputJsonPath: String): Unit = {
  try {
    if (xsdPath.nonEmpty) {
      println(s"Processing XSD: $xsdPath")
      
      // Read XSD file and convert to Spark schema
      val xsdContent = scala.io.Source.fromFile(xsdPath).mkString
      val schema = XSDToSchema.read(xsdContent)
      
      // Convert schema to JSON
      val schemaJson = schema.json
      
      // Write JSON schema to file
      Files.write(Paths.get(outputJsonPath), schemaJson.getBytes(StandardCharsets.UTF_8))
      
      println(s"✓ Schema converted and written to: $outputJsonPath")
      println(s"  Schema has ${schema.fields.length} top-level fields")
      
    } else {
      println(s"⚠ XSD path is empty, skipping conversion")
    }
  } catch {
    case e: Exception => 
      println(s"✗ Error converting XSD $xsdPath: ${e.getMessage}")
      e.printStackTrace()
  }
}

println("=== XSD to JSON Schema Conversion ===")

// Convert each XSD file to JSON schema
if (masterXsdPath.nonEmpty) {
  val filename = masterXsdPath.split("/").last.replace(".xsd", ".json")
  val outputPath = s"${schemaPath}${filename}"
  convertXsdToJson(masterXsdPath, outputPath)
}

import java.io.File

val xsdDir = new File(schemaPath)

println(xsdDir)
if (xsdDir.exists && xsdDir.isDirectory) {
  val xsdFiles = xsdDir.listFiles
    .filter(f => f.isFile && f.getName.endsWith(".xsd"))
    .filterNot(f => 
      f.getName.equalsIgnoreCase(new File(masterXsdPath).getName) || 
      f.getName.equalsIgnoreCase(new File(payloadXsdPath).getName)
    )
  xsdFiles.foreach { file =>
    val filename = file.getName.replace(".xsd", ".json")
    val outputPath = s"${schemaPath}${filename}"
    convertXsdToJson(file.getAbsolutePath, outputPath)
  }
}

if (payloadXsdPath.nonEmpty) {
  val filename = payloadXsdPath.split("/").last.replace(".xsd", ".json")
  val outputPath = s"${schemaPath}${filename}"
  convertXsdToJson(payloadXsdPath, outputPath)
}
println("=== Conversion Complete ===")



In [0]:
from util.xsd_processor import create_specialized_schemas

master_json_path = master_xsd_path.replace(".xsd", ".json")
payload_json_path = payload_xsd_path.replace(".xsd", ".json")

result = create_specialized_schemas(
    master_json_path=master_json_path,
    schema_mappings=schema_mappings,
    row_tag_name=row_tag,
    output_folder=schemas_path,
    validate_schemas=True
)

if result["success"]:
    print(f"✓ Created pyld_schema.json and hdr_pyld_metadata_schema.json")
    pyld_schema_path = result['pyld_schema_path']
    metadata_schema_path = result['metadata_schema_path']
else:
    print(f"✗ Error: {result['error']}")
    pyld_schema_path = None
    metadata_schema_path = None

In [0]:
import os
from util.xsd_processor import create_row_tag_xsd

row_tag_xsd_output = os.path.join(schemas_path, f"row_tag_schema.xsd")

result = create_row_tag_xsd(
    payload_xsd_path=payload_xsd_path,
    row_tag_name=row_tag,
    output_path=row_tag_xsd_output,
    validate_output=True,
)

if result["success"]:
    print(f"✅ Row tag XSD created successfully!")
    print(f"   Output file: {os.path.basename(result['output_path'])}")
    print(f"   Row tag element: {result['row_tag_element']}")
    print(f"   Row tag type: {result['row_tag_type']}")
    
    # Check file size
    file_size = os.path.getsize(row_tag_xsd_output)
    print(f"   File size: {file_size:,} bytes")
    
    row_tag_xsd_path = result['output_path']
    
    print(f"\n💡 This XSD can be used with Spark XML processing:")
    print(f"   spark.read.format('xml').option('row_tag', '{row_tag}').schema(...).load(...)")
else:
    print(f"❌ Row tag XSD creation failed: {result['error']}")
    row_tag_xsd_path = None
